# Lingo Easy — IELTS Speaking Assessment & Adaptive Learning Pipeline

**Google Colab prototype** — record an answer in the browser, run speech/audio analysis, generate an evidence-based IELTS-style assessment, diagnose skill weaknesses, and update a persistent Adaptive Learning Profile.

### What this notebook does
1. Displays a configurable IELTS Speaking question.
2. Records from the browser microphone. **Stop ends the recording; there is no pause/resume.** Maximum duration is configurable.
3. Runs WhisperX transcription + word alignment.
4. Runs OpenPronounce as an experimental phoneme-level pronunciation analyser.
5. Runs Silero VAD + Librosa + Parselmouth acoustic analysis.
6. Aggregates evidence and sends it with the question to a local open-weight Qwen model.
7. Produces four IELTS criterion estimates, overall band estimate, estimated CEFR level, answer relevance, detailed weaknesses and strengths.
8. Updates and exports an Adaptive Learning Profile JSON for future lesson-generation workflows.

### Important limitations
- This is a **prototype**, not an official IELTS score or a validated examiner replacement. Calibrate against trained IELTS raters before using scores in a product.
- OpenPronounce expects reference text. In this notebook, WhisperX's transcript is used as a provisional reference. This creates a circularity risk: ASR errors can affect pronunciation feedback. Treat its score/errors as **experimental evidence**, not ground truth.
- OpenPronounce is calibrated primarily for English read speech; spontaneous speech, Egyptian accents, children, noisy audio and short answers can reduce reliability.
- Audio is stored temporarily in the Colab runtime only and deleted after successful processing. Colab runtime storage is ephemeral, but it is not a privacy guarantee. Do not use identifiable/student production recordings in this prototype without appropriate consent and controls.
- The CEFR mapping is a rough configurable estimate from the speaking band, not an official IELTS-to-CEFR conversion.
- The Qwen model runs locally in the Colab runtime. GPU is strongly recommended. A free Colab T4 may run the quantized model, but availability and speed vary.

In [ ]:
import transformers, whisperx; from transformers import Pipeline; print(transformers.__version__, 'OK Pipeline')

4.57.6 OK Pipeline


## 0. Runtime setup

In Colab choose **Runtime → Change runtime type → T4 GPU** if available. Then run this cell once.

In [ ]:
# Install system dependencies and Python packages.
!apt-get -qq update
!apt-get -qq install -y ffmpeg espeak-ng
!pip -q install -U pip
!pip -q install "whisperx" "openpronounce" "silero-vad" "librosa" "praat-parselmouth" \
    "transformers>=4.51.0" accelerate bitsandbytes sentencepiece soundfile scipy pandas matplotlib

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libpcaudio0:amd64.
(Reading database ... 127324 files and directories currently installed.)
Preparing to unpack .../libpcaudio0_1.2-2build3_amd64.deb ...
Unpacking libpcaudio0:amd64 (1.2-2build3) ...
Selecting previously unselected package libsonic0:amd64.
Preparing to unpack .../libsonic0_0.2.0-13build1_amd64.deb ...
Unpacking libsonic0:amd64 (0.2.0-13build1) ...
Selecting previously unselected package espeak-ng-data:amd64.
Preparing to unpack .../espeak-ng-data_1.51+dfsg-12build1_amd64.deb ...
Unpacking espeak-ng-data:amd64 (1.51+dfsg-12build1) ...
Selecting previously unselected package libespeak-ng1:amd64.
Preparing to unpack .../libespeak-ng1_1.51+dfsg-12build1_amd64.deb ...
Unpacking libespeak-ng1:amd64 (1.51+dfsg-12build1) ...
Selecting previously unsele

## 1. Imports, configuration and current question

Change `SPEAKING_QUESTION` whenever you want to test another question. The question is passed to the evaluator with the transcript and audio evidence, so relevance is assessed against the current prompt.

The recording limit is configurable. The notebook's Stop button ends the recording immediately; it does not pause and resume.

In [ ]:
import os, re, json, time, math, base64, shutil, tempfile, warnings
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import soundfile as sf
import librosa
import matplotlib.pyplot as plt
from IPython.display import display, Audio, HTML, Markdown, clear_output

# ---------------- USER CONFIGURATION ----------------
SPEAKING_QUESTION = "Do you think technology has made people's lives better? Why or why not?"
QUESTION_PART = "Part 3"
TOPIC = "Technology"
RECORDING_MAX_SECONDS = 120
LANGUAGE = "en"

# WhisperX: "base.en" is faster; "small.en" is a stronger default if GPU memory allows.
WHISPERX_MODEL = "small.en"

# Local instruct model. Apache-2.0 model card:
LLM_MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
LLM_MAX_NEW_TOKENS = 2200
LLM_LOAD_4BIT = True

# Optional profile persistence. Set to an existing JSON path in Google Drive if desired.
PROFILE_PATH = "/content/lingo_easy_speaking_profile.json"
STUDENT_ID = "demo_student_001"  # Replace with a non-identifying test ID.

# Approximate IELTS Speaking band -> CEFR mapping (heuristic only; tune/validate separately).
def approximate_cefr_from_ielts_band(band):
    if band is None:
        return "undetermined"
    if band < 4.0: return "A2_or_below"
    if band < 5.5: return "B1"
    if band < 7.0: return "B2"
    if band < 8.5: return "C1"
    return "C2"

print("Question:", SPEAKING_QUESTION)
print("Part:", QUESTION_PART, "| Topic:", TOPIC)
try:
    import torch
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
    print("Device:", DEVICE)
    if DEVICE == "cuda":
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    DEVICE = "cpu"
    print("Torch/device check:", e)

Question: Do you think technology has made people's lives better? Why or why not?
Part: Part 3 | Topic: Technology
Device: cuda
GPU: Tesla T4


## 2. Browser microphone recorder

This uses the browser's `MediaRecorder` API through Colab's output bridge. The UI has **Start** and **Stop** only—no pause/resume. When Stop is clicked, the recording is finalized and returned to Python as a WAV file.

If microphone permission is denied, use the file-upload fallback in the next cell.

In [ ]:
from google.colab import output
from IPython.display import display, HTML

def record_from_microphone(max_seconds=120):
    js = r'''
    async (maxSeconds) => {
      const sleep = (ms) => new Promise(r => setTimeout(r, ms));
      const stream = await navigator.mediaDevices.getUserMedia({audio: true});
      const mime = MediaRecorder.isTypeSupported('audio/webm;codecs=opus')
        ? 'audio/webm;codecs=opus' : 'audio/webm';
      const recorder = new MediaRecorder(stream, {mimeType: mime});
      const chunks = [];
      let stoppedByUser = false;
      let stopTimer = null;

      const root = document.createElement('div');
      root.style.cssText = 'padding:14px;border:1px solid #999;border-radius:12px;max-width:600px';
      const status = document.createElement('div');
      status.textContent = 'Ready. Click Start and answer the question.';
      status.style.cssText = 'margin-bottom:12px;font-weight:600';
      const start = document.createElement('button');
      start.textContent = 'Start recording';
      start.style.cssText = 'padding:10px 16px;margin-right:8px;border-radius:8px';
      const stop = document.createElement('button');
      stop.textContent = 'Stop and finish';
      stop.disabled = true;
      stop.style.cssText = 'padding:10px 16px;border-radius:8px';
      root.append(status, start, stop);
      document.body.appendChild(root);

      const resultPromise = new Promise((resolve, reject) => {
        recorder.ondataavailable = e => { if (e.data && e.data.size) chunks.push(e.data); };
        recorder.onerror = e => reject(e.error || new Error('Recorder error'));
        recorder.onstop = async () => {
          clearTimeout(stopTimer);
          stream.getTracks().forEach(t => t.stop());
          status.textContent = 'Finalizing audio…';
          const blob = new Blob(chunks, {type: mime});
          const arrayBuffer = await blob.arrayBuffer();
          const bytes = new Uint8Array(arrayBuffer);
          let binary = '';
          const step = 0x8000;
          for (let i = 0; i < bytes.length; i += step) {
            binary += String.fromCharCode(...bytes.subarray(i, i + step));
          }
          const dataUrl = 'data:' + mime + ';base64,' + btoa(binary);
          status.textContent = stoppedByUser ? 'Recording stopped.' : 'Maximum duration reached.';
          start.disabled = true; stop.disabled = true;
          resolve(dataUrl);
        };
        start.onclick = () => {
          recorder.start(500);
          start.disabled = true;
          stop.disabled = false;
          status.textContent = 'Recording… Speak naturally. Click Stop and finish when done.';
          stopTimer = setTimeout(() => {
            if (recorder.state === 'recording') {
              stoppedByUser = false;
              recorder.stop();
            }
          }, maxSeconds * 1000);
        };
        stop.onclick = () => {
          if (recorder.state === 'recording') {
            stoppedByUser = true;
            recorder.stop();
          }
        };
      });

      return await resultPromise;
    }
    '''
    data_url = output.eval_js(f"({js})({int(max_seconds)})")
    if not data_url or "," not in data_url:
        raise RuntimeError("No audio was returned from the browser.")
    raw = base64.b64decode(data_url.split(",", 1)[1])
    webm_path = "/content/student_answer.webm"
    wav_path = "/content/student_answer.wav"
    with open(webm_path, "wb") as f:
        f.write(raw)
    import subprocess
    subprocess.run(
        ["ffmpeg", "-y", "-i", webm_path, "-ac", "1", "-ar", "16000", wav_path],
        check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE
    )
    os.remove(webm_path)
    return wav_path

print("Recorder ready. Run the next cell to display the question and start recording.")

Recorder ready. Run the next cell to display the question and start recording.


In [ ]:
display(Markdown(f"### Speaking question — {QUESTION_PART}\n**{SPEAKING_QUESTION}**"))
print(f"Maximum recording time: {RECORDING_MAX_SECONDS} seconds. Press Stop to finish early.")
AUDIO_PATH = record_from_microphone(RECORDING_MAX_SECONDS)
print("Recording saved temporarily:", AUDIO_PATH)
display(Audio(AUDIO_PATH))

### Speaking question — Part 3
**Do you think technology has made people's lives better? Why or why not?**

Maximum recording time: 120 seconds. Press Stop to finish early.
Recording saved temporarily: /content/student_answer.wav


### Optional: upload an existing recording instead

Use this instead of the microphone cell if you already have a WAV/MP3/M4A/WebM recording. It converts the file to mono 16 kHz WAV.

In [ ]:
# Uncomment and run only if you want to upload a file instead of using the microphone.
# from google.colab import files
# uploaded = files.upload()
# input_name = next(iter(uploaded))
# import subprocess
# AUDIO_PATH = "/content/student_answer.wav"
# subprocess.run(["ffmpeg", "-y", "-i", input_name, "-ac", "1", "-ar", "16000", AUDIO_PATH], check=True)
# print("Using:", AUDIO_PATH)

## 3. Audio validation and preprocessing

We keep the original audio only as a temporary processing input. It is not sent to the LLM. The LLM receives structured evidence extracted from it.

In [ ]:
assert os.path.exists(AUDIO_PATH), "Audio file not found. Record or upload an answer first."
audio, sr = sf.read(AUDIO_PATH)
if audio.ndim > 1:
    audio = np.mean(audio, axis=1)
duration_seconds = len(audio) / sr
if duration_seconds < 2:
    raise ValueError("Recording is too short. Please record at least 2 seconds.")
if duration_seconds > RECORDING_MAX_SECONDS + 2:
    warnings.warn("Recording is longer than the configured maximum.")
print(f"Duration: {duration_seconds:.1f}s | Sample rate: {sr} Hz | Samples: {len(audio):,}")

Duration: 83.6s | Sample rate: 16000 Hz | Samples: 1,338,240


## 4. STT — WhisperX transcription and word-level alignment

WhisperX provides transcription and can align words to the audio. Word-level timestamps are useful for estimating speaking rate and pause locations. Alignment can fail for some recordings; the notebook preserves the transcript if alignment is unavailable.

In [ ]:
import torch, whisperx

compute_type = "float16" if torch.cuda.is_available() else "int8"
asr_device = "cuda" if torch.cuda.is_available() else "cpu"

print("Loading WhisperX model:", WHISPERX_MODEL, "| device:", asr_device, "| compute:", compute_type)
asr_model = whisperx.load_model(
    WHISPERX_MODEL, asr_device, compute_type=compute_type, language=LANGUAGE
)
asr_result = asr_model.transcribe(AUDIO_PATH, batch_size=8 if asr_device == "cuda" else 1)
detected_language = asr_result.get("language", LANGUAGE)
segments = asr_result.get("segments", [])
raw_transcript = " ".join(s.get("text", "").strip() for s in segments).strip()

aligned_result = None
word_segments = []
try:
    align_model, align_metadata = whisperx.load_align_model(
        language_code=detected_language, device=asr_device
    )
    aligned_result = whisperx.align(
        segments, align_model, align_metadata, AUDIO_PATH, asr_device,
        return_char_alignments=False
    )
    word_segments = aligned_result.get("word_segments", [])
except Exception as e:
    print("Word alignment unavailable; keeping segment-level transcript. Reason:", repr(e))

transcript = " ".join(raw_transcript.split())
print("=== WHISPERX TRANSCRIPT ===")
print(transcript if transcript else "[No speech recognized]")
print("\nDetected language:", detected_language)
print("Aligned words:", len(word_segments))
if word_segments:
    display(pd.DataFrame(word_segments).head(30))

Loading WhisperX model: small.en | device: cuda | compute: float16


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocabulary.txt: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.bin:   0%|          | 0.00/484M [00:00<?, ?B/s]

... (2 outputs left out)

Downloading: "https://download.pytorch.org/torchaudio/models/wav2vec2_fairseq_base_ls960_asr_ls960.pth" to /root/.cache/torch/hub/checkpoints/wav2vec2_fairseq_base_ls960_asr_ls960.pth


100%|██████████| 360M/360M [00:06<00:00, 56.4MB/s]


=== WHISPERX TRANSCRIPT ===
So, hello. I think that technology has made people's lives better. Why? Because technology like AI, internet, computer and etc. help people increase their productivity, help them achieve more and finish their works and the tasks easier and more efficient and faster than ever before. So, it made people's lives become better. But on the other hand, as it improved people's lives, it also affected people's lives in a negative way. For example, AI helped people increase productivity but it helped lazy students to cheat in exams or it made them stop searching for the answers or for information and make them more lazy.

Detected language: en
Aligned words: 104


,word,start,end,score
0,"So,",1.507,1.888,0.995
1,hello.,2.428,2.849,0.888
2,I,5.271,5.331,0.999
3,think,5.391,5.671,0.995
4,that,5.811,6.071,0.920
5,technology,6.552,7.272,0.923
6,has,7.613,7.773,0.862
7,made,7.813,8.013,0.780
8,people's,8.053,8.433,0.832
9,lives,8.493,8.754,0.829


In [ ]:
import traceback; exec("try:\n from transformers import Pipeline\n print('Pipeline OK')\nexcept BaseException:\n traceback.print_exc()\n import transformers.pipelines")

Pipeline OK


## 5. Pronunciation — OpenPronounce (experimental)

OpenPronounce compares the recording against expected text. Because this is an open-ended IELTS answer, there is no single correct answer. We therefore pass the WhisperX transcript as a **provisional reference**.

This is a useful prototype signal, but not a fully independent pronunciation score. A transcript error can contaminate the comparison. The output must be interpreted cautiously, and low-confidence flags should not become confirmed student weaknesses.

In [ ]:
openpronounce_result = None
openpronounce_error = None

try:
    from openpronounce import load_audio, compare_audio_with_text
    if transcript:
        sound = load_audio(AUDIO_PATH)
        openpronounce_result = compare_audio_with_text(sound, transcript, lang="en")
        print("OpenPronounce score (experimental, not IELTS band):",
              openpronounce_result.get("score"))
        errors = (openpronounce_result.get("differences") or {}).get("errors", [])
        print("Detected pronunciation differences:", len(errors))
        if errors:
            display(pd.DataFrame(errors))
        else:
            print("No word-level differences returned.")
    else:
        openpronounce_error = "Skipped: WhisperX returned an empty transcript."
except Exception as e:
    openpronounce_error = repr(e)
    print("OpenPronounce could not run:", openpronounce_error)
    print("The remaining pipeline can continue; pronunciation evidence will be marked unavailable.")

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

loading configuration file preprocessor_config.json from cache at /root/.cache/huggingface/hub/models--facebook--wav2vec2-large-960h/snapshots/bdeaacdf88f7a155f50a2704bc967aa81fbbb2ab/preprocessor_config.json
Feature extractor Wav2Vec2FeatureExtractor {
  "do_normalize": true,
  "feature_extractor_type": "Wav2Vec2FeatureExtractor",
  "feature_size": 1,
  "padding_side": "right",
  "padding_value": 0.0,
  "return_attention_mask": false,
  "sampling_rate": 16000
}



tokenizer_config.json:   0%|          | 0.00/163 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--facebook--wav2vec2-large-960h/snapshots/bdeaacdf88f7a155f50a2704bc967aa81fbbb2ab/config.json
Model config Wav2Vec2Config {
  "activation_dropout": 0.1,
  "adapter_attn_dim": null,
  "adapter_kernel_size": 3,
  "adapter_stride": 2,
  "add_adapter": false,
  "apply_spec_augment": true,
  "architectures": [
    "Wav2Vec2ForCTC"
  ],
  "attention_dropout": 0.1,
  "bos_token_id": 1,
  "classifier_proj_size": 256,
  "codevector_dim": 256,
  "contrastive_logits_temperature": 0.1,
  "conv_bias": false,
  "conv_dim": [
    512,
    512,
    512,
    512,
    512,
    512,
    512
  ],
  "conv_kernel": [
    10,
    3,
    3,
    3,
    3,
    2,
    2
  ],
  "conv_stride": [
    5,
    2,
    2,
    2,
    2,
    2,
    2
  ],
  "ctc_loss_reduction": "sum",
  "ctc_zero_infinity": false,
  "diversity_loss_weight": 0.1,
  "do_stable_layer_norm": false,
  "eos_token_id": 2,
  "feat_extract_activation": "gelu"

vocab.json:   0%|          | 0.00/291 [00:00<?, ?B/s]

... (15 outputs left out)

loading weights file pytorch_model.bin from cache at /root/.cache/huggingface/hub/models--facebook--wav2vec2-lv-60-espeak-cv-ft/snapshots/ae45363bf3413b374fecd9dc8bc1df0e24c3b7f4/pytorch_model.bin
Attempting to create safetensors variant
Safetensors PR exists


model.safetensors:   0%|          | 0.00/1.26G [00:00<?, ?B/s]

OpenPronounce score (experimental, not IELTS band): 54.87
Detected pronunciation differences: 5


,position,word,expected,actual,actual_word,phone_distance,confidence,phones
0,4,that,ðæt,ðædə,,2,0.973,"[{'expected': 'ð', 'heard': 'ð', 'confidence':..."
1,34,the,ðə,,,2,0.476,"[{'expected': 'ð', 'heard': '', 'confidence': ..."
2,84,cheat,tʃit,ʃipt,,2,0.754,"[{'expected': 'tʃ', 'heard': 'ʃ', 'confidence'..."
3,87,or,oɹ,æ,,1,0.987,"[{'expected': 'oɹ', 'heard': 'æ', 'confidence'..."
4,100,make,meɪk,meɪdz,,2,1.000,"[{'expected': 'm', 'heard': 'm', 'confidence':..."


## 6. Audio analysis — Silero VAD + Librosa + Parselmouth

VAD detects speech and non-speech regions. We derive pause duration, speech duration, speech rate and filler counts. Librosa and Parselmouth provide acoustic measurements such as RMS energy and pitch statistics.

**Interpretation warning:** pitch variation is not an automatic measure of good intonation; speaking rate is not a fluency score; and silence may reflect normal planning. These are supporting signals, not direct IELTS grades.

In [ ]:
# --- Silero VAD ---
vad_segments = []
vad_status = "unavailable"
try:
    from silero_vad import load_silero_vad, read_audio, get_speech_timestamps
    vad_model = load_silero_vad()
    wav_tensor = read_audio(AUDIO_PATH, sampling_rate=16000)
    vad_raw = get_speech_timestamps(
        wav_tensor, vad_model, sampling_rate=16000, return_seconds=True
    )
    vad_segments = [{"start": float(x["start"]), "end": float(x["end"])} for x in vad_raw]
    vad_status = "silero_vad"
except Exception as e:
    print("Silero VAD unavailable; using energy-based fallback:", repr(e))
    # Simple fallback: 20 ms RMS frames; threshold is relative to recording level.
    frame = int(0.02 * sr)
    rms = []
    for i in range(0, len(audio), frame):
        chunk = audio[i:i+frame]
        rms.append(float(np.sqrt(np.mean(chunk**2) + 1e-12)))
    rms = np.asarray(rms)
    threshold = max(float(np.max(rms)) * 0.12, 0.008)
    active = rms > threshold
    # Fill gaps of <= 2 frames to reduce fragmentation.
    for i in range(1, len(active)-2):
        if active[i-1] and active[i+1]:
            active[i] = True
    starts = np.where(np.diff(np.r_[False, active, False].astype(int)) == 1)[0]
    ends = np.where(np.diff(np.r_[False, active, False].astype(int)) == -1)[0]
    vad_segments = [
        {"start": float(s*frame/sr), "end": float(min(e*frame, len(audio))/sr)}
        for s, e in zip(starts, ends) if (e-s)*frame/sr >= 0.10
    ]
    vad_status = "energy_fallback"

speech_duration = sum(max(0, x["end"]-x["start"]) for x in vad_segments)
pause_gaps = []
for prev, nxt in zip(vad_segments, vad_segments[1:]):
    gap = max(0.0, nxt["start"] - prev["end"])
    if gap > 0:
        pause_gaps.append(gap)

long_pause_threshold = 1.0
long_pauses = [p for p in pause_gaps if p >= long_pause_threshold]
word_count = len(re.findall(r"\b[\w']+\b", transcript))
speech_rate_wpm = word_count / max(speech_duration, 1e-6) * 60
articulation_rate_wpm = word_count / max(speech_duration, 1e-6) * 60

fillers = re.findall(r"\b(?:um+|uh+|erm+|ah+)\b", transcript.lower())
filler_counts = dict(Counter(fillers))

# --- Librosa features ---
y, sr_lib = librosa.load(AUDIO_PATH, sr=16000, mono=True)
rms_values = librosa.feature.rms(y=y)[0]
rms_mean = float(np.mean(rms_values)) if len(rms_values) else None
zcr_mean = float(np.mean(librosa.feature.zero_crossing_rate(y=y)[0]))

# --- Parselmouth / Praat pitch and intensity ---
pitch_stats = {"mean_hz": None, "std_hz": None, "voiced_frame_ratio": None}
intensity_stats = {"mean_db": None, "std_db": None}
try:
    import parselmouth
    snd = parselmouth.Sound(y, sampling_frequency=16000)
    pitch = snd.to_pitch()
    pitch_values = pitch.selected_array["frequency"]
    voiced = pitch_values[pitch_values > 0]
    if len(voiced):
        pitch_stats = {
            "mean_hz": float(np.mean(voiced)),
            "std_hz": float(np.std(voiced)),
            "voiced_frame_ratio": float(len(voiced)/max(len(pitch_values), 1))
        }
    intensity = snd.to_intensity()
    intensity_values = intensity.values[0]
    intensity_stats = {
        "mean_db": float(np.mean(intensity_values)),
        "std_db": float(np.std(intensity_values))
    }
except Exception as e:
    print("Parselmouth analysis unavailable:", repr(e))

audio_features = {
    "duration_seconds": round(duration_seconds, 3),
    "vad_method": vad_status,
    "speech_duration_seconds": round(speech_duration, 3),
    "estimated_silence_seconds": round(max(0, duration_seconds-speech_duration), 3),
    "pause_count": len(pause_gaps),
    "mean_pause_seconds": round(float(np.mean(pause_gaps)), 3) if pause_gaps else 0.0,
    "long_pause_threshold_seconds": long_pause_threshold,
    "long_pause_count": len(long_pauses),
    "long_pause_durations_seconds": [round(x, 3) for x in long_pauses],
    "word_count_from_transcript": word_count,
    "speech_rate_wpm_estimate": round(speech_rate_wpm, 1),
    "articulation_rate_wpm_estimate": round(articulation_rate_wpm, 1),
    "filler_counts_from_transcript": filler_counts,
    "rms_energy_mean": rms_mean,
    "zero_crossing_rate_mean": zcr_mean,
    "pitch": pitch_stats,
    "intensity": intensity_stats
}

print("=== AUDIO ANALYSIS ===")
display(pd.DataFrame([audio_features]).T.rename(columns={0:"value"}))
if vad_segments:
    display(pd.DataFrame(vad_segments).head(25))

/usr/local/lib/python3.13/dist-packages/silero_vad/utils_vad.py:202: UserWarning: torchaudio.sox_effects.sox_effects.apply_effects_file has been deprecated. This deprecation is part of a large refactoring effort to transition TorchAudio into a maintenance phase. Please see https://github.com/pytorch/audio/issues/3902 for more information. It will be removed from the 2.9 release. 
  return torchaudio.sox_effects.apply_effects_file(path, effects=effects)
/usr/local/lib/python3.13/dist-packages/torchaudio/_backend/utils.py:213: UserWarning: In 2.9, this function's implementation will be changed to use torchaudio.load_with_torchcodec` under the hood. Some parameters like ``normalize``, ``format``, ``buffer_size``, and ``backend`` will be ignored. We recommend that you port your code to rely directly on TorchCodec's decoder instead: https://docs.pytorch.org/torchcodec/stable/generated/torchcodec.decoders.AudioDecoder.html#torchcodec.decoders.AudioDecoder.
  warnings.warn(
/usr/local/lib/pyt

=== AUDIO ANALYSIS ===


,value
duration_seconds,83.64
vad_method,silero_vad
speech_duration_seconds,73.0
estimated_silence_seconds,10.64
pause_count,17
mean_pause_seconds,0.5
long_pause_threshold_seconds,1.0
long_pause_count,3
long_pause_durations_seconds,"[1.7, 1.1, 1.9]"
word_count_from_transcript,104


,start,end
0,1.1,3.8
1,4.3,9.6
2,10.0,15.8
3,16.5,17.6
4,17.7,29.2
5,29.5,32.7
6,34.4,42.0
7,42.1,50.1
8,50.9,58.6
9,58.7,59.9


## 7. Evidence aggregation

This cell combines the question, transcript, word timings, pronunciation output and acoustic features into one JSON object. It does **not** convert acoustic features directly into IELTS bands.

In [ ]:
def json_safe(obj):
    if obj is None or isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {str(k): json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [json_safe(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    try:
        return float(obj)
    except Exception:
        return str(obj)

pronunciation_evidence = {"status": "unavailable"}
if openpronounce_result is not None:
    diffs = openpronounce_result.get("differences") or {}
    pronunciation_evidence = {
        "status": "experimental_provisional_reference",
        "reference_text_source": "WhisperX transcript",
        "score_0_100": json_safe(openpronounce_result.get("score")),
        "transcription_by_openpronounce": json_safe(openpronounce_result.get("transcribe")),
        "phoneme_error_rate": json_safe(diffs.get("phoneme_error_rate")),
        "word_error_rate": json_safe(diffs.get("word_error_rate")),
        "errors": json_safe(diffs.get("errors", [])),
        "heard_phones": json_safe(diffs.get("heard_phones", [])),
        "expected_phones": json_safe(diffs.get("expected_phones", [])),
        "prosody": json_safe(openpronounce_result.get("prosody", {})),
        "limitations": [
            "WhisperX transcript is used as provisional expected text.",
            "ASR errors can contaminate pronunciation comparison.",
            "Do not treat this score as an IELTS Pronunciation band."
        ]
    }
elif openpronounce_error:
    pronunciation_evidence["error"] = openpronounce_error

evidence = {
    "attempt": {
        "student_id": STUDENT_ID,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "question": SPEAKING_QUESTION,
        "part": QUESTION_PART,
        "topic": TOPIC,
        "audio_duration_seconds": round(duration_seconds, 3)
    },
    "transcription": {
        "text": transcript,
        "language": detected_language,
        "word_segments": json_safe(word_segments),
        "raw_segments": json_safe(segments)
    },
    "pronunciation": pronunciation_evidence,
    "audio_features": json_safe(audio_features),
    "quality_flags": {
        "empty_transcript": not bool(transcript),
        "short_answer_under_10_words": word_count < 10,
        "very_short_recording_under_8_seconds": duration_seconds < 8,
        "pronunciation_reference_is_asr_transcript": openpronounce_result is not None,
        "vad_method": vad_status
    }
}

print(json.dumps(evidence, ensure_ascii=False, indent=2)[:14000])

{
  "attempt": {
    "student_id": "demo_student_001",
    "timestamp_utc": "2026-10-30T19:32:41.393892+00:00",
    "question": "Do you think technology has made people's lives better? Why or why not?",
    "part": "Part 3",
    "topic": "Technology",
    "audio_duration_seconds": 83.64
  },
  "transcription": {
    "text": "So, hello. I think that technology has made people's lives better. Why? Because technology like AI, internet, computer and etc. help people increase their productivity, help them achieve more and finish their works and the tasks easier and more efficient and faster than ever before. So, it made people's lives become better. But on the other hand, as it improved people's lives, it also affected people's lives in a negative way. For example, AI helped people increase productivity but it helped lazy students to cheat in exams or it made them stop searching for the answers or for information and make them more lazy.",
    "language": "en",
    "word_segments": [
      

## 8. Load the local IELTS Speaking Evaluation LLM — Qwen3 4B Instruct

The model is downloaded from Hugging Face and runs in this Colab runtime. The model card lists Apache-2.0 licensing. A GPU is recommended; 4-bit loading reduces VRAM requirements.

If loading fails on a free runtime, switch to a smaller quantized model or use a hosted inference endpoint separately (that would no longer be fully local/free).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

llm_device = "cuda" if torch.cuda.is_available() else "cpu"
print("Loading:", LLM_MODEL_ID, "| device:", llm_device)

quant_config = None
if llm_device == "cuda" and LLM_LOAD_4BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )

tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_ID, trust_remote_code=True)
model_kwargs = {"device_map": "auto", "trust_remote_code": True}
if quant_config is not None:
    model_kwargs["quantization_config"] = quant_config
else:
    model_kwargs["torch_dtype"] = torch.float16 if llm_device == "cuda" else torch.float32

llm = AutoModelForCausalLM.from_pretrained(LLM_MODEL_ID, **model_kwargs)
llm.eval()
print("LLM loaded.")

Loading: Qwen/Qwen3-4B-Instruct-2507 | device: cuda


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

loading file vocab.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/vocab.json
loading file merges.txt from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/merges.txt
loading file tokenizer.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/tokenizer.json
loading file added_tokens.json from cache at None
loading file special_tokens_map.json from cache at None
loading file tokenizer_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/tokenizer_config.json
loading file chat_template.jinja from cache at None
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


... (9 outputs left out)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/238 [00:00<?, ?B/s]

loading configuration file generation_config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen3-4B-Instruct-2507/snapshots/cdbee75f17c01a7cc42f958dc650907174af0554/generation_config.json
Generate config GenerationConfig {
  "bos_token_id": 151643,
  "do_sample": true,
  "eos_token_id": [
    151645,
    151643
  ],
  "pad_token_id": 151643,
  "temperature": 0.7,
  "top_k": 20,
  "top_p": 0.8
}

Could not locate the custom_generate/generate.py inside Qwen/Qwen3-4B-Instruct-2507.


LLM loaded.


## 9. IELTS evaluator system prompt and evaluation request

The evaluator must:
- Assess **all four official IELTS Speaking criteria**, each on the 0–9 band scale.
- Assess answer relevance and development against the actual question.
- Distinguish language ability from topic knowledge and avoid penalising a reasonable opinion.
- Use audio evidence for pronunciation/fluency only where available.
- Never infer phoneme errors from transcript alone.
- Identify specific skill weaknesses using stable Skill IDs.
- Return evidence, confidence and recommended learning objectives.
- Return valid JSON only.

The model's scores are estimates. A single answer does not equal a full IELTS Speaking test result.

In [ ]:
SYSTEM_PROMPT = r'''
You are Lingo Easy's IELTS Speaking assessment and learner-diagnosis engine.
You are NOT an official IELTS examiner. Produce careful practice estimates, not official scores.

Assess the answer using the four IELTS Speaking criteria:
1. fluency_and_coherence
2. lexical_resource
3. grammatical_range_and_accuracy
4. pronunciation

Use the IELTS public band-descriptor concepts:
- Fluency/coherence: continuity, hesitation, repetition, self-correction, logical sequencing,
  relevance, development and cohesion.
- Lexical resource: range, precision, collocation, paraphrase and flexibility.
- Grammar: range and accuracy of structures, error frequency and impact.
- Pronunciation: intelligibility, phonological control, stress, rhythm and intonation.
Do not require a British or American accent. Do not penalise an Egyptian accent merely for being Egyptian.
Do not reward speed by itself. Normal thinking pauses are not automatically errors.

CRITICAL EVIDENCE RULES:
- The question is part of the input. Judge whether the response answers it, stays on topic,
  develops ideas and supports claims. Do not require one "correct" opinion.
- Transcript supports grammar, vocabulary, coherence and relevance, but cannot prove pronunciation.
- Pronunciation tool output here may be based on an ASR-generated provisional reference. Treat it as
  experimental. Do not convert its 0-100 score directly to an IELTS band.
- Use phoneme/word errors only as low-to-medium confidence signals unless independently supported.
- Audio metrics (rate, pauses, pitch, energy) are supporting signals, not direct IELTS grades.
- Do not infer emotions, personality, intelligence or identity from voice.
- Do not invent examples, errors, evidence, timestamps or phoneme substitutions.
- If evidence is missing or weak, say "insufficient_evidence" and lower confidence.
- A short answer may be insufficient to estimate some criteria reliably. Flag this explicitly.
- Distinguish an observed issue in this answer from a recurring learner weakness. One answer alone
  should generally produce "observed", not "confirmed".
- Avoid treating ASR recognition mistakes as the student's grammar mistakes. Mark uncertain transcript
  spans as uncertain rather than correcting them confidently.

SKILL TAXONOMY — use these IDs when supported:
Pronunciation:
pronunciation.individual_phonemes, pronunciation.vowel_sounds, pronunciation.consonant_clusters,
pronunciation.word_stress, pronunciation.sentence_stress, pronunciation.intonation,
pronunciation.connected_speech, pronunciation.rhythm, pronunciation.intelligibility
Fluency:
fluency.long_pauses, fluency.filled_pauses, fluency.repetition, fluency.self_correction,
fluency.speech_rate, fluency.continuity, fluency.idea_transitions
Grammar:
grammar.tenses, grammar.subject_verb_agreement, grammar.articles, grammar.prepositions,
grammar.singular_plural, grammar.sentence_structure, grammar.complex_sentences,
grammar.conditionals, grammar.relative_clauses
Vocabulary:
vocabulary.range, vocabulary.word_choice, vocabulary.collocations, vocabulary.repetition,
vocabulary.paraphrasing, vocabulary.topic_vocabulary, vocabulary.idiomatic_usage
Coherence:
coherence.relevance, coherence.idea_development, coherence.logical_organization,
coherence.supporting_examples, coherence.linking, coherence.avoid_irrelevance

Return exactly one JSON object (no Markdown fences) with this schema:
{
  "assessment_type": "single_answer_practice_estimate",
  "scores": {
    "fluency_and_coherence": {"band": 0.0, "confidence": 0.0, "rationale": ""},
    "lexical_resource": {"band": 0.0, "confidence": 0.0, "rationale": ""},
    "grammatical_range_and_accuracy": {"band": 0.0, "confidence": 0.0, "rationale": ""},
    "pronunciation": {"band": 0.0, "confidence": 0.0, "rationale": ""},
    "overall_band_estimate": 0.0,
    "cefr_estimate": "A2_or_below|B1|B2|C1|C2|undetermined",
    "cefr_mapping_note": "rough estimate only; not an official conversion"
  },
  "answer_analysis": {
    "relevance": "direct|mostly_relevant|partly_relevant|off_topic|insufficient_evidence",
    "relevance_confidence": 0.0,
    "answered_question": true,
    "idea_development": "limited|adequate|well_developed|insufficient_evidence",
    "summary": ""
  },
  "diagnostic_report": {
    "identified_weaknesses": [{
      "skill_id": "",
      "category": "pronunciation|fluency|grammar|vocabulary|coherence",
      "status": "observed|possible",
      "severity": "low|medium|high",
      "confidence": 0.0,
      "occurrences": 1,
      "evidence": [{"type": "transcript|audio_metric|pronunciation_tool", "detail": ""}],
      "learning_objective": "",
      "recommended_exercise": ""
    }],
    "identified_strengths": [{
      "skill_id": "",
      "category": "",
      "confidence": 0.0,
      "evidence": []
    }],
    "priority_learning_objectives": [],
    "insufficient_evidence": []
  },
  "feedback": {
    "fluency_and_coherence": "",
    "lexical_resource": "",
    "grammar": "",
    "pronunciation": "",
    "answer_relevance": ""
  },
  "transcript_corrections": [{
    "original": "",
    "suggested_correction": "",
    "skill_id": "",
    "explanation": ""
  }],
  "next_practice_recommendations": []
}
Use numeric band values in 0.5 increments from 0 to 9. Overall is the arithmetic average of four criterion bands rounded to nearest half band (practice estimate only). If evidence is insufficient, do not fabricate certainty; use low confidence and explain.
'''

def build_user_prompt(evidence_obj):
    return (
        "Evaluate this single IELTS Speaking practice answer. The current question and all "
        "available evidence are supplied below as JSON. The question may vary between attempts.\n"
        "Return only the required JSON object.\n\n"
        + json.dumps(evidence_obj, ensure_ascii=False)
    )

user_prompt = build_user_prompt(evidence)
print("System prompt and user prompt prepared.")

System prompt and user prompt prepared.


In [ ]:
def generate_llm_response(system_prompt, user_prompt, max_new_tokens=2200):
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt}
    ]
    prompt_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
    )
    inputs = tokenizer(prompt_text, return_tensors="pt").to(llm.device)
    with torch.inference_mode():
        generated = llm.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            eos_token_id=tokenizer.eos_token_id
        )
    new_tokens = generated[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

llm_raw_output = generate_llm_response(
    SYSTEM_PROMPT, user_prompt, max_new_tokens=LLM_MAX_NEW_TOKENS
)
print(llm_raw_output[:16000])

{
  "assessment_type": "single_answer_practice_estimate",
  "scores": {
    "fluency_and_coherence": {"band": 6.5, "confidence": 0.85, "rationale": "The speaker maintains a continuous flow with minimal hesitation, using clear transitions like 'on the other hand' to link ideas. There are three long pauses (1.7s, 1.1s, 1.9s), which are acceptable in Part 3 and do not disrupt coherence. The response logically develops a balanced view: starting with positive impacts of technology, then introducing a counter-argument about negative effects. Ideas are relevant and well-sequenced, though the development of the negative example could be deeper."},
    "lexical_resource": {"band": 6.0, "confidence": 0.9, "rationale": "The vocabulary is appropriate and relevant to the topic. Key terms such as 'productivity', 'efficiency', 'lazy students', 'cheating', and 'information' are used accurately. Collocations like 'increase productivity', 'finish tasks easier and faster', and 'stop searching for answers

## 10. Parse, validate and display the assessment

The parser attempts to extract JSON even if the model accidentally adds surrounding text. It checks the expected top-level fields before continuing.

In [ ]:
# Free GPU memory held by earlier stages, then send the LLM a compact evidence object (full word/segment dumps blew past GPU memory).
import gc
for _n in ["asr_model", "align_model", "sound", "vad_model", "wav_tensor"]: globals().pop(_n, None)
gc.collect(); torch.cuda.empty_cache()
compact_evidence = {"attempt": evidence["attempt"], "transcription": {"text": evidence["transcription"]["text"], "language": evidence["transcription"]["language"], "low_confidence_words": [{"word": w.get("word"), "start": w.get("start"), "score": round(w["score"], 2)} for w in word_segments if w.get("score") is not None and w["score"] < 0.6]}, "pronunciation": {k: v for k, v in evidence["pronunciation"].items() if k in ("status", "reference_text_source", "score_0_100", "phoneme_error_rate", "word_error_rate", "limitations", "error")} | {"errors": evidence["pronunciation"].get("errors", [])[:15]}, "audio_features": evidence["audio_features"], "quality_flags": evidence["quality_flags"]}
user_prompt = build_user_prompt(compact_evidence)
print("GPU free GB:", round(torch.cuda.mem_get_info()[0] / 1e9, 1), "| prompt tokens:", len(tokenizer(SYSTEM_PROMPT + user_prompt).input_ids))

GPU free GB: 5.9 | prompt tokens: 2663


In [ ]:
def extract_json(text):
    text = text.strip()
    # Remove accidental markdown fences.
    text = re.sub(r"^```(?:json)?\s*", "", text)
    text = re.sub(r"\s*```$", "", text)
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start, end = text.find("{"), text.rfind("}")
        if start >= 0 and end > start:
            return json.loads(text[start:end+1])
        raise

assessment = extract_json(llm_raw_output)
required = ["assessment_type", "scores", "answer_analysis", "diagnostic_report", "feedback"]
missing = [k for k in required if k not in assessment]
if missing:
    raise ValueError(f"LLM JSON is missing required keys: {missing}")

scores = assessment.get("scores", {})
criterion_keys = [
    "fluency_and_coherence", "lexical_resource",
    "grammatical_range_and_accuracy", "pronunciation"
]
bands = []
for key in criterion_keys:
    item = scores.get(key, {})
    band = item.get("band") if isinstance(item, dict) else None
    if isinstance(band, (int, float)) and 0 <= band <= 9:
        bands.append(float(band))

if len(bands) == 4:
    average = sum(bands) / 4
    calculated_overall = round(average * 2) / 2
    scores["overall_band_estimate"] = calculated_overall
    scores["cefr_estimate"] = approximate_cefr_from_ielts_band(calculated_overall)
    scores["cefr_mapping_note"] = "Heuristic estimate only; not an official conversion."

print("=== IELTS SPEAKING PRACTICE ASSESSMENT ===")
display(pd.DataFrame([
    {"Criterion": "Fluency & Coherence", "Band": scores.get("fluency_and_coherence", {}).get("band"), "Confidence": scores.get("fluency_and_coherence", {}).get("confidence")},
    {"Criterion": "Lexical Resource", "Band": scores.get("lexical_resource", {}).get("band"), "Confidence": scores.get("lexical_resource", {}).get("confidence")},
    {"Criterion": "Grammar", "Band": scores.get("grammatical_range_and_accuracy", {}).get("band"), "Confidence": scores.get("grammatical_range_and_accuracy", {}).get("confidence")},
    {"Criterion": "Pronunciation", "Band": scores.get("pronunciation", {}).get("band"), "Confidence": scores.get("pronunciation", {}).get("confidence")}
]))
print("Overall band estimate:", scores.get("overall_band_estimate"))
print("Estimated CEFR:", scores.get("cefr_estimate"))
print("\nAnswer relevance:", assessment.get("answer_analysis", {}).get("relevance"))
print("Answered question:", assessment.get("answer_analysis", {}).get("answered_question"))
print("\nSummary:", assessment.get("answer_analysis", {}).get("summary"))

=== IELTS SPEAKING PRACTICE ASSESSMENT ===


,Criterion,Band,Confidence
0,Fluency & Coherence,6.5,0.85
1,Lexical Resource,6.0,0.90
2,Grammar,6.0,0.80
3,Pronunciation,5.5,0.70


Overall band estimate: 6.0
Estimated CEFR: B2

Answer relevance: direct
Answered question: True

Summary: The student provides a balanced response to the question, discussing both positive and negative impacts of technology. The argument is logically structured with clear topic progression and relevant examples. The development of the negative consequence (students cheating) is sufficient and relevant.


## 11. Weaknesses, strengths and learning recommendations

These are attempt-level findings. They are not yet persistent profile truths. The profile updater below combines findings across attempts and marks repeated issues as recurring.

In [ ]:
diagnosis = assessment.get("diagnostic_report", {})
weaknesses = diagnosis.get("identified_weaknesses", [])
strengths = diagnosis.get("identified_strengths", [])

print("=== IDENTIFIED WEAKNESSES ===")
if weaknesses:
    display(pd.DataFrame([{
        "Skill ID": w.get("skill_id"),
        "Category": w.get("category"),
        "Status": w.get("status"),
        "Severity": w.get("severity"),
        "Confidence": w.get("confidence"),
        "Occurrences": w.get("occurrences"),
        "Learning objective": w.get("learning_objective"),
        "Exercise": w.get("recommended_exercise")
    } for w in weaknesses]))
else:
    print("No weaknesses were confidently identified.")

print("\n=== IDENTIFIED STRENGTHS ===")
if strengths:
    display(pd.DataFrame([{
        "Skill ID": s.get("skill_id"),
        "Category": s.get("category"),
        "Confidence": s.get("confidence")
    } for s in strengths]))
else:
    print("No strengths were confidently identified.")

print("\n=== PRIORITY LEARNING OBJECTIVES ===")
for item in diagnosis.get("priority_learning_objectives", []):
    print("•", item)
print("\n=== FEEDBACK ===")
for k, v in assessment.get("feedback", {}).items():
    print(f"\n{k.replace('_',' ').title()}:\n{v}")

=== IDENTIFIED WEAKNESSES ===


,Skill ID,Category,Status,Severity,Confidence,Occurrences,Learning objective,Exercise
0,pronunciation.vowel_sounds,pronunciation,observed,medium,0.8,3,Improve vowel sound production in common words...,"Practice minimal pairs: 'or' vs 'our', 'the' v..."
1,lexical_resource.collocations,vocabulary,observed,low,0.8,1,Expand vocabulary with more precise collocatio...,"Use topic-specific collocation lists (e.g., 'i..."
2,grammar.complex_sentences,grammar,possible,low,0.6,1,"Introduce more complex sentence types (e.g., c...",Practice writing and speaking with complex sen...



=== IDENTIFIED STRENGTHS ===


,Skill ID,Category,Confidence
0,fluency_and_coherence.idea_transitions,coherence,0.90
1,lexical_resource.topic_vocabulary,vocabulary,0.85



=== PRIORITY LEARNING OBJECTIVES ===
• Improve pronunciation of common vowels and consonant clusters in academic contexts.
• Expand lexical range with more precise and varied expressions related to technology and education.
• Develop complex sentence structures to enhance grammatical range and academic tone.

=== FEEDBACK ===

Fluency And Coherence:
Good flow and logical organization. Consider adding a brief example to support the negative impact (e.g., 'In schools, students might rely on AI to write essays instead of learning how to write themselves.') to strengthen development.

Lexical Resource:
Use more specific and varied vocabulary (e.g., 'digital tools', 'cognitive load', 'information overload') instead of repeating 'help' and 'better'.

Grammar:
Avoid repetitive sentence patterns. Try using a mix of present simple, present perfect, and conditional structures to show deeper analysis.

Pronunciation:
Focus on clear vowel sounds in words like 'or', 'the', 'cheat', and 'make'. Pra

## 12. Adaptive Learning Profile — create or update persistent JSON

The profile is separate from the single-answer assessment. It accumulates attempts and tracks:
- skill ID and category
- number of attempts where the weakness was observed
- average confidence
- latest and previous occurrence
- severity and learning objective
- current status: observed / recurring / confirmed-candidate

**Prototype rule:** a weakness is marked `recurring` after appearing in at least 2 separate attempts with confidence ≥ 0.60. It is not automatically marked confirmed; confirmation should come from targeted practice or additional evidence.

In [ ]:
def load_profile(path):
    if os.path.exists(path):
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return {
        "schema_version": "1.0",
        "student_id": STUDENT_ID,
        "created_at": datetime.now(timezone.utc).isoformat(),
        "updated_at": None,
        "attempt_count": 0,
        "speaking_summary": {},
        "skill_states": {},
        "attempt_history": []
    }

def update_profile(profile, assessment_obj, evidence_obj):
    now = datetime.now(timezone.utc).isoformat()
    diag = assessment_obj.get("diagnostic_report", {})
    scores_obj = assessment_obj.get("scores", {})
    attempt_id = f"speaking_{profile.get('attempt_count', 0)+1:04d}"

    attempt_record = {
        "attempt_id": attempt_id,
        "timestamp_utc": now,
        "question": evidence_obj["attempt"]["question"],
        "part": evidence_obj["attempt"]["part"],
        "transcript": evidence_obj["transcription"]["text"],
        "scores": scores_obj,
        "answer_analysis": assessment_obj.get("answer_analysis", {}),
        "weakness_skill_ids": [],
        "strength_skill_ids": []
    }

    skill_states = profile.setdefault("skill_states", {})
    for w in diag.get("identified_weaknesses", []):
        skill_id = w.get("skill_id")
        if not skill_id:
            continue
        attempt_record["weakness_skill_ids"].append(skill_id)
        state = skill_states.setdefault(skill_id, {
            "skill_id": skill_id,
            "category": w.get("category"),
            "observed_attempts": 0,
            "confidence_values": [],
            "first_seen": now,
            "last_seen": now,
            "severity": w.get("severity", "medium"),
            "status": "observed",
            "evidence_history": [],
            "learning_objectives": []
        })
        conf = float(w.get("confidence", 0) or 0)
        state["observed_attempts"] += 1
        state["confidence_values"].append(conf)
        state["last_seen"] = now
        state["severity"] = w.get("severity", state.get("severity", "medium"))
        if w.get("learning_objective") and w["learning_objective"] not in state["learning_objectives"]:
            state["learning_objectives"].append(w["learning_objective"])
        state["evidence_history"].append({
            "attempt_id": attempt_id,
            "confidence": conf,
            "evidence": w.get("evidence", []),
            "occurrences_in_attempt": w.get("occurrences", 1)
        })
        reliable_attempts = sum(1 for x in state["confidence_values"] if x >= 0.60)
        state["status"] = "recurring" if reliable_attempts >= 2 else "observed"
        state["average_confidence"] = round(float(np.mean(state["confidence_values"])), 3)

    for s in diag.get("identified_strengths", []):
        if s.get("skill_id"):
            attempt_record["strength_skill_ids"].append(s["skill_id"])

    profile["attempt_count"] = profile.get("attempt_count", 0) + 1
    profile["attempt_history"].append(attempt_record)
    profile["updated_at"] = now
    profile["speaking_summary"] = {
        "latest_overall_band_estimate": scores_obj.get("overall_band_estimate"),
        "latest_cefr_estimate": scores_obj.get("cefr_estimate"),
        "latest_criterion_bands": {
            k: (scores_obj.get(k) or {}).get("band") for k in criterion_keys
        },
        "latest_answer_relevance": (assessment_obj.get("answer_analysis") or {}).get("relevance"),
        "priority_learning_objectives": diag.get("priority_learning_objectives", [])
    }
    return profile

profile = load_profile(PROFILE_PATH)
profile = update_profile(profile, assessment, evidence)

# Rank learning priorities: recurring first, then severity and confidence.
severity_weight = {"high": 3, "medium": 2, "low": 1}
ranked_skills = sorted(
    profile["skill_states"].values(),
    key=lambda x: (
        x.get("status") == "recurring",
        severity_weight.get(x.get("severity"), 1),
        x.get("average_confidence", 0),
        x.get("observed_attempts", 0)
    ),
    reverse=True
)
profile["learning_priorities"] = [
    {
        "skill_id": x["skill_id"],
        "category": x.get("category"),
        "status": x.get("status"),
        "severity": x.get("severity"),
        "average_confidence": x.get("average_confidence"),
        "learning_objectives": x.get("learning_objectives", [])
    }
    for x in ranked_skills[:10]
]

with open(PROFILE_PATH, "w", encoding="utf-8") as f:
    json.dump(profile, f, ensure_ascii=False, indent=2)

print("Profile updated:", PROFILE_PATH)
print("Attempts stored:", profile["attempt_count"])
display(pd.DataFrame(profile["learning_priorities"]))

Profile updated: /content/lingo_easy_speaking_profile.json
Attempts stored: 1


,skill_id,category,status,severity,average_confidence,learning_objectives
0,pronunciation.vowel_sounds,pronunciation,observed,medium,0.8,[Improve vowel sound production in common word...
1,lexical_resource.collocations,vocabulary,observed,low,0.8,[Expand vocabulary with more precise collocati...
2,grammar.complex_sentences,grammar,observed,low,0.6,"[Introduce more complex sentence types (e.g., ..."


## 13. Export results for future lesson generation

Two separate JSON files are exported:
1. `speaking_assessment_<attempt>.json` — this attempt's evidence and assessment.
2. `lingo_easy_speaking_profile.json` — cumulative profile for the student.

The second file is the one a future lesson-generation pipeline can consume. It contains stable Skill IDs and prioritized learning objectives.

In [ ]:
attempt_export = {
    "evidence": evidence,
    "assessment": assessment
}
ATTEMPT_EXPORT_PATH = "/content/lingo_easy_speaking_assessment.json"

with open(ATTEMPT_EXPORT_PATH, "w", encoding="utf-8") as f:
    json.dump(json_safe(attempt_export), f, ensure_ascii=False, indent=2)

with open(PROFILE_PATH, "w", encoding="utf-8") as f:
    json.dump(json_safe(profile), f, ensure_ascii=False, indent=2)

print("Saved files:")
print(ATTEMPT_EXPORT_PATH)
print(PROFILE_PATH)

# Download both JSON files to your computer.
from google.colab import files
files.download(ATTEMPT_EXPORT_PATH)
files.download(PROFILE_PATH)

Saved files:
/content/lingo_easy_speaking_assessment.json
/content/lingo_easy_speaking_profile.json


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 14. Cleanup temporary audio

Run after all analysis is complete. This deletes the temporary WAV from the current Colab runtime. The notebook does not store or export the recording.

In [ ]:
if os.path.exists(AUDIO_PATH):
    os.remove(AUDIO_PATH)
    print("Temporary audio deleted:", AUDIO_PATH)
else:
    print("No temporary audio file found.")

## 15. Reusing the profile in Lingo Easy lesson generation

*   List item
*   List item



Example payload for a future lesson generator. Do not send the entire raw transcript history unless the generator needs it; normally send the profile's prioritized skill states and the learner's current CEFR/course context.

```python
lesson_generation_input = {
    "student_id": profile["student_id"],
    "target_skill": "speaking",
    "estimated_cefr": profile["speaking_summary"].get("latest_cefr_estimate"),
    "learning_priorities": profile.get("learning_priorities", []),
    "instruction": (
        "Generate lessons and practice tasks that target the listed weaknesses. "
        "Respect each skill_id and do not teach already-mastered content unnecessarily."
    )
}
```

### Recommended next validation step
Run 20–30 varied answers (different IELTS Parts, topics, lengths and Egyptian-English proficiency levels). Compare the system's criterion scores and diagnosed weaknesses against qualified IELTS-speaking raters. Track false positives in pronunciation and ASR-related errors separately. Only after calibration should the scoring be exposed as a reliable learner-facing estimate.